# Preprocessing and Pipelines

## What you'll learn
- Why raw data must be prepared before modelling
- Filling missing values (`SimpleImputer`), encoding categories (`OneHotEncoder`),
  and scaling numbers (`StandardScaler`)
- `ColumnTransformer`: different steps for different columns
- `Pipeline`: preprocessing and model as one object, with no data leakage
- Tuning settings with `GridSearchCV`, and saving a trained model to disk

## The raw data

Earlier lessons dropped messy rows to keep things simple. Here we load the data **as it
is**: some values are missing and some regions are in lower case.

In [ ]:
import pandas as pd

df = pd.read_csv("data/customer_accounts.csv")
print(df.isna().sum()[df.isna().sum() > 0])
print(df["region"].value_counts())

## Why not just clean everything first?

Filling a missing value with "the average" uses information from the whole dataset,
including the test rows. The model then gets a small peek at the test set, and its test
score looks better than it will be on real new data. This is **data leakage**.

The fix is to learn every preprocessing step (the averages, the categories, the scaling)
from the **training** data only, and apply it unchanged to the test data. A `Pipeline`
does this automatically.

In [ ]:
from sklearn.model_selection import train_test_split

df["region"] = df["region"].str.title()   # fixing text case is safe: it learns nothing

NUMERIC = ["years_as_customer", "on_contract", "orders_last_year", "avg_order_value",
           "discount_pct", "support_tickets", "avg_days_to_pay", "spend_last_year"]
CATEGORICAL = ["segment", "region"]

X = df[NUMERIC + CATEGORICAL]
y = df["paid_late"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

## Preprocessing for each column type

- **Numeric columns:** fill gaps with the median, then scale.
- **Category columns:** fill gaps with the most common value, then one-hot encode.
  `handle_unknown="ignore"` means a category never seen in training does not crash the model.

`ColumnTransformer` applies each recipe to its own columns and joins the results.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_steps = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
category_steps = Pipeline([
    ("fill", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("numeric", numeric_steps, NUMERIC),
    ("category", category_steps, CATEGORICAL),
])

## The full pipeline

Add the model as the last step. Now `fit` learns the preprocessing and the model from the
training data together, and `predict` applies exactly the same steps to new data.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression()),
])
pipeline.fit(X_train, y_train)
print(f"Test ROC AUC: {roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1]):.3f}")
pipeline

The last line shows the pipeline as a diagram in Jupyter. Click the boxes to see each step.

## Tuning with GridSearchCV

Models have settings (**hyperparameters**) such as a forest's number of trees or a
logistic regression's regularisation strength `C`. `GridSearchCV` tries every combination
with cross-validation on the training data and keeps the best. Step names are joined with
double underscores: `model__C` means "the `C` setting of the step called `model`".

In [ ]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(pipeline, param_grid={"model__C": [0.01, 0.1, 1, 10]},
                      cv=5, scoring="roc_auc")
search.fit(X_train, y_train)
print("best setting:", search.best_params_)
print(f"best cross-validated ROC AUC: {search.best_score_:.3f}")
print(f"test ROC AUC: {roc_auc_score(y_test, search.predict_proba(X_test)[:, 1]):.3f}")

## Save and reload the model

A trained pipeline can be saved to a file and loaded later (in another notebook or a
scheduled job) to score new accounts, with all its preprocessing included. `joblib`
installs with scikit-learn.

**Only load model files you created or fully trust.** Loading a `joblib` file can run code
stored inside it, so never open one received from an unknown source.

In [ ]:
import os
import joblib

os.makedirs("models", exist_ok=True)
joblib.dump(search.best_estimator_, "models/late_payment_model.joblib")

loaded = joblib.load("models/late_payment_model.joblib")   # our own file, so safe to load
new_accounts = pd.DataFrame([
    {"years_as_customer": 1, "on_contract": 0, "orders_last_year": 70, "avg_order_value": 11000,
     "discount_pct": 11, "support_tickets": 6, "avg_days_to_pay": 58, "spend_last_year": 770000,
     "segment": "Distributor", "region": "Latin America"},
    {"years_as_customer": 12, "on_contract": 1, "orders_last_year": 35, "avg_order_value": 8500,
     "discount_pct": None, "support_tickets": 1, "avg_days_to_pay": 24, "spend_last_year": 297500,
     "segment": "Hospital", "region": "Europe"},
])
print(loaded.predict_proba(new_accounts)[:, 1].round(2))

The second account has a missing discount; the saved pipeline fills it with the training
median automatically.

## Practice

1. Replace `LogisticRegression()` in the pipeline with a `RandomForestClassifier` and
   compare the test ROC AUC. (Forests do not need scaling, but it does no harm.)
2. Change the numeric fill strategy to `"mean"`. Does the score change?
3. Use `GridSearchCV` on the forest pipeline to try `model__n_estimators` of 100 and 300
   and `model__min_samples_leaf` of 1, 5, and 20.
4. Score a new account from a region the model has never seen (for example
   `"Middle East"`). What happens, and why?

In [ ]:
# Your practice code here

## Summary

- Real data needs gaps filled, categories encoded, and numbers scaled.
- Learning those steps from all the data leaks test information; learn them from training data only.
- `ColumnTransformer` applies different steps to different columns.
- A `Pipeline` bundles preprocessing and model, so `fit` and `predict` stay consistent.
- `GridSearchCV` tunes settings with cross-validation; `joblib` saves the trained pipeline.

## Practice Solutions

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data/customer_accounts.csv")
df["region"] = df["region"].str.title()
NUMERIC = ["years_as_customer", "on_contract", "orders_last_year", "avg_order_value",
           "discount_pct", "support_tickets", "avg_days_to_pay", "spend_last_year"]
CATEGORICAL = ["segment", "region"]
X, y = df[NUMERIC + CATEGORICAL], df["paid_late"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


def make_pipeline_with(model, numeric_fill="median"):
    preprocess = ColumnTransformer([
        ("numeric", Pipeline([("fill", SimpleImputer(strategy=numeric_fill)),
                              ("scale", StandardScaler())]), NUMERIC),
        ("category", Pipeline([("fill", SimpleImputer(strategy="most_frequent")),
                               ("encode", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL),
    ])
    return Pipeline([("preprocess", preprocess), ("model", model)])


def test_auc(p):
    return roc_auc_score(y_test, p.predict_proba(X_test)[:, 1])


# 1.
forest = make_pipeline_with(RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))
print(f"forest {test_auc(forest.fit(X_train, y_train)):.3f}")

# 2.
for fill in ["median", "mean"]:
    p = make_pipeline_with(LogisticRegression(), numeric_fill=fill).fit(X_train, y_train)
    print(fill, f"{test_auc(p):.3f}")

# 3.
search = GridSearchCV(forest, {"model__n_estimators": [100, 300],
                               "model__min_samples_leaf": [1, 5, 20]}, cv=5, scoring="roc_auc")
search.fit(X_train, y_train)
print(search.best_params_, f"{search.best_score_:.3f}")

# 4. handle_unknown="ignore" encodes the unseen region as all zeros, so it still scores.
new = X_test.iloc[[0]].assign(region="Middle East")
print(search.predict_proba(new)[:, 1].round(2))